# Notebook to configure and run the SED

### The goal of the project is to receive structures in methods and dictionaries that indicate the operation of the emergency system. Internally, the simulation components were created, and Simpy is used as the simulation engine.
### In this way, this notebook will indicate all the methods and objects necessary for the creation of the main class and will execute a process, similar to the Rodada_Upa_Single.py file.


### Each method and data strucute have a docstring with descripitions

In [1]:
import copy
import statistics

import pandas as pd
import simpy
import random
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt
from random import expovariate, seed, normalvariate
from scipy import stats
from plotly.subplots import make_subplots
import plotly.graph_objects as go


In [2]:
from Modelos import Simulacao, CorridaSimulacao

## Auxiliar Methods

In [3]:
"""
A method that indicates the probability distributions of each process in the simulation.
The key of dict needs to be the same of the other structures.
"""


def distribuicoes_base(processo, slot="None"):
    coef_processos = 60  # Conversão para minutos!!
    coef_chegadas = 60
    coef_checkin = 60
    dados = {
        "Chegada": expovariate(0.0029),
        "Ficha": max(0.5, random.lognormvariate(0.460, 0.576)),
        "Triagem": max(0.6, random.lognormvariate(-4.454, 8.946)),
        "Clínico": max(4.53, random.weibullvariate(6.878, 2.832)),
        "Pediatra": max(5.34, random.gauss(14.022, 5.966)),
        "Raio-x": 5 * coef_chegadas,  # Cincominutos
        "Eletro": 12 * coef_chegadas,
        "Exame de Urina": 2 * coef_chegadas,
        "Exame de Sangue": 3 * coef_chegadas,
        "Análise de Sangue Externo": 0.25
        * 60
        * coef_chegadas,  # Quatrohoras,masreduziprameiaho
        "Análise de Sangue Interno": 0.1 * 60 * coef_chegadas,
        "Análise de Urina": 2 * 60 * coef_chegadas,
        "Aplicar Medicação": random.triangular(
            10 * coef_chegadas, 60 * coef_chegadas, 40 * coef_chegadas
        ),
        "Tomar Medicação": random.gauss(35.350, 2.443),
    }

    return dados[processo]

In [4]:
"""
Method that calculate the probabilities of each decision in the simulation.
"""
def calcula_distribuicoes_prob():
    def calcula(dados):
        inicio = 0
        list_aux = []
        for dado in dados:
            list_aux.append([inicio, inicio + dado[1], dado[0]])
            inicio = inicio + dado[1]
        return list_aux

        # 1 - clinico e 2 -  pediatra

    classificacao_clinico_pediatra = [["Clínico", 0.78], ["Pediatra", 0.22]]
    # 5 - menos grave e 1 - mais grave
    classificacao_prioridade = [[4, 0.033], [3, 0.70129], [2, 0.150], [1, 0.117]]

    # saida do sistema após o clinico
    decisao_apos_clinico = [
        ["Saída", 0.4],
        ["Aplicar Medicação", 0.2],
        ["Raio-x", 0.1],
        ["Eletro", 0.1],
        ["Exame de Urina", 0.1],
        ["Exame de Sangue", 0.1],
    ]

    decisao_apos_pediatra = [
        ["Saída", 0.4],
        ["Aplicar Medicação", 0.2],
        ["Raio-x", 0.1],
        ["Eletro", 0.1],
        ["Exame de Urina", 0.1],
        ["Exame de Sangue", 0.1],
    ]

    decisao_apos_medicacao = [
        ["Saída", 0.4],
        [
            "medico",
            0.2,
        ],
        ["Raio-x", 0.1],
        ["Eletro", 0.1],
        ["Exame de Urina", 0.1],
        ["Exame de Sangue", 0.1],
    ]

    decisao_apos_urina = [
        ["medico", 0.7],
        ["Raio-x", 0.1],
        ["Eletro", 0.1],
        ["Exame de Sangue", 0.1],
    ]

    decisao_apos_exame_sangue = [
        ["medico", 0.7],
        ["Raio-x", 0.1],
        ["Eletro", 0.1],
        ["Exame de Urina", 0.1],
    ]

    decisao_apos_raio_x = [
        ["medico", 0.7],
        ["Exame de Sangue", 0.1],
        ["Eletro", 0.1],
        ["Exame de Urina", 0.1],
    ]

    decisao_apos_eletro = [
        ["medico", 0.7],
        ["Exame de Sangue", 0.1],
        ["Raio-x", 0.1],
        ["Exame de Urina", 0.1],
    ]

    # Decisao para tempo de espera do resultado do exame de sangue!!!!
    analise_de_sangue = [[0.5 * 60 * 60, 0.5], [0.25 * 60 * 60, 0.5]]

    analise_urina = [[0.25 * 60 * 60, 1]]

    dict_atr = {
        "decide_atendimento": calcula(classificacao_clinico_pediatra),
        "prioridade": calcula(classificacao_prioridade),
        "decisao_apos_clinico": calcula(decisao_apos_clinico),
        "decisao_apos_pediatra": calcula(decisao_apos_pediatra),
        "decisao_apos_raio_x": calcula(decisao_apos_raio_x),
        "decisao_apos_eletro": calcula(decisao_apos_eletro),
        "decisao_apos_urina": calcula(decisao_apos_urina),
        "decisao_apos_exame_sangue": calcula(decisao_apos_exame_sangue),
        "decisao_apos_medicacao": calcula(decisao_apos_medicacao),
        "tempo_resultado_exame_sangue": calcula(analise_de_sangue),
        "tempo_resultado_exame_urina": calcula(analise_urina),
    }

    return dict_atr

## Setting the simulation parameters

In [5]:
"""
A structure that indicates the next step in each process, 
whether it's a decision based on probability or another process.
For proper functioning, every process needs to have a defined next step.
Drawing a parallel with the Arena software, these are the process blocks.


keys are process and values is a process or decision defined in method calcula_distribuicoes_prob
"""

ordem_processo = {
        "Ficha": "Triagem",
        "Triagem": ["decide_atendimento"],
        "Clínico": ["decisao_apos_clinico"],
        "Pediatra": ["decisao_apos_pediatra"],
        "Aplicar Medicação": "Tomar Medicação",
        "Tomar Medicação": ["decisao_apos_medicacao"],
        "Exame de Urina": ["decisao_apos_urina"],
        "Exame de Sangue": ["decisao_apos_exame_sangue"],
        "Análise de Urina": "medico",
        "Raio-x": ["decisao_apos_raio_x"],
        "Eletro": ["decisao_apos_eletro"],
    }

In [6]:
"""
Data structure that indicates which resources are needed in each process.
If the process does not consume any resources, pass an empty list.
Keys are process and values is a list with the resourcers names.

"""


necessidade_recursos = {
        "Ficha": ["Secretária"],
        "Triagem": ["Enfermeira de Triagem"],
        "Clínico": ["Clínico"],
        "Pediatra": ["Pediatra"],
        "Raio-x": ["Raio-x"],
        "Exame de Urina": [],
        "Exame de Sangue": ["Técnica de Enfermagem"],
        "Análise de Sangue Externo": [],
        "Análise de Sangue Interno": [],
        "Análise de Urina": [],
        "Aplicar Medicação": ["Técnica de Enfermagem", "Espaço para tomar Medicação"],
        "Tomar Medicação": [],
        "Eletro": ["Eletro"],
    }

In [7]:
"""
Data structure that indicates which resources are released in each process.
If the process does not release any resources, pass an empty list.
"""


liberacao_recursos = {
        "Ficha": ["Secretária"],
        "Triagem": ["Enfermeira de Triagem"],
        "Clínico": ["Clínico"],
        "Pediatra": ["Pediatra"],
        "Raio-x": ["Raio-x"],
        "Exame de Urina": [],
        "Exame de Sangue": ["Técnica de Enfermagem"],
        "Análise de Sangue Externo": [],
        "Análise de Sangue Interno": [],
        "Análise de Urina": [],
        "Aplicar Medicação": ["Técnica de Enfermagem"],
        "Tomar Medicação": ["Espaço para tomar Medicação"],
        "Eletro": ["Eletro"],
    }

In [8]:
"""
A data structure that indicates which properties are assigned to entities in each process.
A key is a process, and a value is a decision.
"""

atribuicoes_processo = {
        "Triagem": "prioridade",
        "Exame de Sangue": "tempo_resultado_exame_sangue",
        "Exame de Urina": "tempo_resultado_exame_urina",
    }

In [9]:
"""
Data structure that indicates which process evaluates the priorities of the entities.
"""


prioridades = {
        "Ficha": None,
        "Triagem": None,
        "Clínico": "prioridade",
        "Pediatra": "prioridade",
    }

In [10]:
"""
A structure that indicates which resources will be used in the simulation and their respective capacities.
"""


recursos_base = {
        "Secretária": [2, False],
        "Enfermeira de Triagem": [2, False],
        "Clínico": [3, True],
        "Pediatra": [2, True],
        "Raio-x": [1, True],
        "Eletro": [1, True],
        "Técnica de Enfermagem": [2, True],
        "Espaço para tomar Medicação": [8, True],
        "Default_Aguarda_Medicacao": [100000, False],
    } 

In [11]:
"""
Time settings, number of replications, warmup

"""

warmup = 5 * 86400
replicacoes = 30
tempo = 24 * 60 * 60 * 30 * 1
distribuicoes_probabilidade = calcula_distribuicoes_prob()

## Instance Simulation Class

In [12]:
simulacao_base = Simulacao(
    distribuicoes=distribuicoes_base,
    imprime=False,
    recursos=recursos_base,
    dist_prob=distribuicoes_probabilidade,
    tempo=tempo,
    necessidade_recursos=necessidade_recursos,
    ordem_processo=ordem_processo,
    atribuicoes=atribuicoes_processo,
    liberacao_recurso=liberacao_recursos,
    warmup=0,
)

## Instance Simulation Run Class

In [13]:
CorridaSimulacao_base = CorridaSimulacao(
        replicacoes=replicacoes,
        simulacao=simulacao_base,
        duracao_simulacao=tempo,
        periodo_warmup=warmup,
        plota_histogramas=True,
    )

## Run Simulation

In [ ]:
CorridaSimulacao_base.roda_simulacao()